# 内置 Timer 的同步式计时语义

> 对应博客篇目：《[性能剖析（00）：度量口径——MFU、HFU、Timer 与 goodput，先把尺子做准](https://lrypcy.github.io/2026/09/29/profiling-00-metrics/)》§4
>
> 代码逐字摘自博客仓库 `tools/profiling_bench/timer_semantics.py`（[lrypcy/ipynbs](https://github.com/lrypcy/ipynbs)）。纯 CPU，不需要 numpy。

## 这个 notebook 回答什么

Megatron 内置的 `Timer` 用的不是 CUDA Event，而是 `torch.cuda.synchronize()` + `time.time()`（`megatron/core/timers.py:143` / `:156`）。同步点会等**所有流排空**，于是**边界处还在飞的工作被算进了被测量的区间**。

而「在飞的工作」正是你辛苦做重叠的那部分。所以：

> **重叠做得越好，内置 timer 的读数膨胀得越厉害。优化的方向和指标的方向反了。**

本 notebook 分四步：

| 步骤 | 问题 |
|:---|:---|
| 1 | 同步点数量 × 通信占比 → 计时膨胀多少 |
| 2 | 多 rank 的 min/max 能不能相加 |
| 3 | `filter(x > 0.0)` 静默丢了谁 |
| 4 | `reset()` 不清 `_active_time` 会让占空比变成什么 |

## 第 0 步 · 环境

只需要标准库（`threading` / `time` / `statistics`）。本 notebook 的实验一用**真线程 + sleep** 模拟设备侧耗时，语义与 CUDA 的异步发射一致，但线程调度会带来额外开销（每 chunk 多少由脚本自己打印，见实验一输出）。

脚本会把每 chunk 的调度开销显式打出来，所以看趋势与比值即可，不要纠结绝对值。

In [1]:
from __future__ import annotations
import argparse
import statistics
import threading
import time

## 第 1 步 · 同步式计时对重叠的破坏

**这步在做什么**：构造两条流——compute 流逐 chunk 计算，comm 流的第 $i$ 个 chunk 依赖 compute 流第 $i$ 个 chunk 完成（1F1B 反向重叠的语义）。chunk 时长 10 ms，共 8 个 chunk。

然后对比两种计时口径：

- **时间线口径**：一次发射到底，不在中间插同步点 → 这是「真实时间线」
- **同步式计时**：每 `phase_len` 个 chunk 调一次 `stop()`（排空所有流）再 `start()` → 这是内置 timer 的语义

膨胀量的解析式就是边界处被迫串行掉的通信：读数 = $$L(T_c + T_m)$$，真实 = $$L \cdot T_c + T_m$$，差 $$(L-1)T_m$$。

$T_m$ 越大，重叠越值得做，读数膨胀也越厉害。

In [2]:
# ---- 摘自 tools/profiling_bench/timer_semantics.py ----
class _ComputeWorker(threading.Thread):
    """compute 流：逐 chunk 执行，每个 chunk 完成后放一个事件通知 comm 流。"""

    def __init__(self, n_chunks: int, chunk_ms: float):
        super().__init__(daemon=True)
        self.n_chunks = n_chunks
        self.chunk_s = chunk_ms / 1000.0
        self.done = [threading.Event() for _ in range(n_chunks)]
        self.busy_s = 0.0


    def run(self) -> None:
        for i in range(self.n_chunks):
            t0 = time.perf_counter()
            time.sleep(self.chunk_s)
            self.busy_s += time.perf_counter() - t0
            self.done[i].set()



class _CommWorker(threading.Thread):
    """comm 流：第 i 个 chunk 依赖 compute 流第 i 个 chunk 完成（反向重叠语义）。"""

    def __init__(self, compute: _ComputeWorker, chunk_ms: float):
        super().__init__(daemon=True)
        self.compute = compute
        self.chunk_s = chunk_ms / 1000.0
        self.busy_s = 0.0

    def run(self) -> None:
        for i in range(self.compute.n_chunks):
            self.compute.done[i].wait()
            if self.chunk_s <= 0:
                continue
            t0 = time.perf_counter()
            time.sleep(self.chunk_s)
            self.busy_s += time.perf_counter() - t0



def _run_async(n_chunks: int, tc_ms: float, tm_ms: float):
    """一次发射到底：重叠不受干扰。返回 (makespan_ms, compute_busy_ms, comm_busy_ms)。"""
    c = _ComputeWorker(n_chunks, tc_ms)
    m = _CommWorker(c, tm_ms)
    t0 = time.perf_counter()
    c.start()
    m.start()
    c.join()
    m.join()
    return (time.perf_counter() - t0) * 1000.0, c.busy_s * 1000.0, m.busy_s * 1000.0



def _run_synced(n_chunks: int, tc_ms: float, tm_ms: float, phase_len: int):
    """每 phase_len 个 chunk 调一次 stop()（同步排空），再调 start()。

    返回 (各阶段读数之和 ms, 各阶段读数列表 ms)。
    """
    readings = []
    for beg in range(0, n_chunks, phase_len):
        k = min(phase_len, n_chunks - beg)
        c = _ComputeWorker(k, tc_ms)
        m = _CommWorker(c, tm_ms)
        t0 = time.perf_counter()          # stop() 已排空 -> start() -> 读时钟
        c.start()
        m.start()
        c.join()
        m.join()
        readings.append((time.perf_counter() - t0) * 1000.0)  # stop() 排空 -> 读时钟
    return sum(readings), readings



def experiment_sync_inflation(n_chunks=8, tc_ms=10.0, repeats=3):
    """实验一：同步点数量 × 通信占比 -> 计时膨胀。"""
    tm_values = [0.0, 2.0, 5.0, 10.0]
    phase_lens = [(n_chunks, "无中间同步点（≈时间线口径）"),
                  (n_chunks // 2, "2 个阶段"),
                  (1, "每 chunk 一个阶段（L 段）")]

    rows = []
    for tm_ms in tm_values:
        best_async = min(_run_async(n_chunks, tc_ms, tm_ms)[0] for _ in range(repeats))
        row = {"tm_ms": tm_ms, "async_ms": best_async, "sync": {}}
        for plen, label in phase_lens:
            total = min(_run_synced(n_chunks, tc_ms, tm_ms, plen)[0]
                        for _ in range(repeats))
            row["sync"][plen] = (total, label)
        rows.append(row)

    print("=" * 78)
    print("实验一  同步式计时对重叠的破坏（compute chunk = %.0f ms，共 %d 个 chunk）"
          % (tc_ms, n_chunks))
    print("=" * 78)
    print("真实时间线（一次发射到底）:")
    print("  chunk 时长 -> " + "  ".join("Tm=%4.1fms" % r["tm_ms"] for r in rows))
    print("  makespan   -> " + "  ".join("%8.1f" % r["async_ms"] for r in rows))
    ovh = (rows[0]["async_ms"] - n_chunks * tc_ms) / n_chunks
    print("  （每 chunk 的线程调度开销 ≈ %.2f ms，所以绝对值比解析式系统性偏大，看趋势与比值）"
          % ovh)
    print()
    print("同步式计时读数（各阶段求和）与膨胀率:")
    for plen, label in phase_lens:
        tot = "  ".join("%8.1f" % r["sync"][plen][0] for r in rows)
        inf = "  ".join("%7.0f%%" % (100.0 * (r["sync"][plen][0] / r["async_ms"] - 1))
                        for r in rows)
        print("  [%s]" % label)
        print("     求和 " + tot)
        print("     膨胀 " + inf)
    print()

    # 逐 chunk 同步那档的解析值对照
    print("解析对照（L 段每段都同步）: 读数 = L*(Tc+Tm)，真实 = L*Tc+Tm")
    for r in rows:
        L, tc, tm = n_chunks, tc_ms, r["tm_ms"]
        print("  Tm=%4.1fms -> 解析 %6.1f / 实测 %6.1f   真实 %6.1f"
              % (tm, L * (tc + tm), r["sync"][1][0], r["async_ms"]))
    print()
    return rows



experiment_sync_inflation()

实验一  同步式计时对重叠的破坏（compute chunk = 10 ms，共 8 个 chunk）
真实时间线（一次发射到底）:
  chunk 时长 -> Tm= 0.0ms  Tm= 2.0ms  Tm= 5.0ms  Tm=10.0ms
  makespan   ->     94.8      97.5      98.8     107.1
  （每 chunk 的线程调度开销 ≈ 1.85 ms，所以绝对值比解析式系统性偏大，看趋势与比值）

同步式计时读数（各阶段求和）与膨胀率:
  [无中间同步点（≈时间线口径）]
     求和     93.0      91.6      98.2     106.9
     膨胀      -2%       -6%       -1%       -0%
  [2 个阶段]
     求和     94.5     100.4     106.5     119.2
     膨胀      -0%        3%        8%       11%
  [每 chunk 一个阶段（L 段）]
     求和     89.2     112.9     142.3     188.3
     膨胀      -6%       16%       44%       76%

解析对照（L 段每段都同步）: 读数 = L*(Tc+Tm)，真实 = L*Tc+Tm
  Tm= 0.0ms -> 解析   80.0 / 实测   89.2   真实   94.8
  Tm= 2.0ms -> 解析   96.0 / 实测  112.9   真实   97.5
  Tm= 5.0ms -> 解析  120.0 / 实测  142.3   真实   98.8
  Tm=10.0ms -> 解析  160.0 / 实测  188.3   真实  107.1



[{'tm_ms': 0.0,
  'async_ms': 94.81716700247489,
  'sync': {8: (93.02933301660232, '无中间同步点（≈时间线口径）'),
   4: (94.48437500395812, '2 个阶段'),
   1: (89.21633401769213, '每 chunk 一个阶段（L 段）')}},
 {'tm_ms': 2.0,
  'async_ms': 97.50800000620075,
  'sync': {8: (91.60420898115262, '无中间同步点（≈时间线口径）'),
   4: (100.37137399194762, '2 个阶段'),
   1: (112.91766699287109, '每 chunk 一个阶段（L 段）')}},
 {'tm_ms': 5.0,
  'async_ms': 98.76433399040252,
  'sync': {8: (98.17158299847506, '无中间同步点（≈时间线口径）'),
   4: (106.52204099460505, '2 个阶段'),
   1: (142.28945793001913, '每 chunk 一个阶段（L 段）')}},
 {'tm_ms': 10.0,
  'async_ms': 107.07179197925143,
  'sync': {8: (106.94545900332741, '无中间同步点（≈时间线口径）'),
   4: (119.22070899163373, '2 个阶段'),
   1: (188.32354198093526, '每 chunk 一个阶段（L 段）')}}]

> **这是一条与直觉相反的规律**：通信占计算比例越高（`Tm` 越大、越值得藏），「每 chunk 一个阶段」那一档的膨胀率越夸张。优化的方向和指标的方向反了。
>
> **正确做法**：要「阶段耗时」就用时间线口径——CUDA Event 打点，或从 nsys / torch.profiler 的 trace 里读该区间的实际跨度。要「端到端步时」就只在外层打一对点，中间不要插同步点。

### 关于这一格数字的抖动

输出里「每 chunk 的线程调度开销」那一行是**实测值**，它随机器负载变化很大（本 notebook
这次记录到的值见上面输出）。它会同时抬高分子和分母，所以：

- **膨胀率的单调性是稳的**——`Tm` 越大膨胀越大，这个趋势每次运行都成立；
- **膨胀率的具体数值不恒定**——调度开销越大，`L·(T_c+T_m)` 里被 `L` 放大的那一项占比越高，膨胀率读数会偏低。

博客正文引用的是另一次运行的输出，与这里的**同趋势、绝对值不同**。想复现正文的绝对值，
请在机器空闲时重跑，并以脚本自己打印的调度开销那一行判断可信度。

实验二、三、四是**确定性计算**，没有这个抖动，数字与博客正文逐位一致。

## 第 2 步 · 多 rank 的 min/max 不是同一张卡

**这步在做什么**：`timers.py:318` 的 `_get_global_min_max_time()` 对每个阶段分别取各 rank 的 min 与 max。**min 与 max 可能来自不同的卡。**

构造两种形态做对照（确定性合成，不用随机数）：

- **形态 A（掉队卡固定）**：rank 3 在每个阶段都慢 1.6× → 各阶段 max 落在同一张卡上
- **形态 B（短板轮转）**：每个阶段换一张卡慢 → 各阶段 max 落在不同的卡上

两种形态对「逐阶段相加」的影响完全不同。

In [3]:
# ---- 摘自 tools/profiling_bench/timer_semantics.py ----
def _synth_ranks(n_ranks: int, n_phases: int, base_ms: float, jitter_ms: float,
                 straggler_rank: int, straggler_factor: float, rotate_spike: bool):
    """确定性合成各 rank 各阶段耗时（不用随机数，避免复现差异）。

    rotate_spike=False：掉队卡在**每个阶段**都慢 -> 各阶段 max 落在同一张卡上
    rotate_spike=True ：短板**每阶段换一张卡** -> 各阶段 max 落在不同的卡上
    这两种形态对「逐阶段相加」的影响完全不同，是实验二要对比的重点。
    """
    table = []
    for r in range(n_ranks):
        row = []
        for p in range(n_phases):
            frac = ((r * 7919 + p * 104729) % 1000) / 1000.0
            v = base_ms + jitter_ms * frac
            if r == straggler_rank:
                v *= straggler_factor
            if rotate_spike and r == (p * 3 + 1) % n_ranks:
                v += jitter_ms * 6.0
            row.append(v)
        table.append(row)
    return table



def experiment_minmax_additivity(n_ranks=8, n_phases=6, base_ms=100.0, jitter_ms=20.0,
                                 straggler_rank=3, straggler_factor=1.6):
    """实验二：`_get_global_min_max_time()` 报的 min/max 不可加。"""
    print("=" * 78)
    print("实验二  多 rank 的 min/max 不是同一张卡，各项 max 相加没有物理含义")
    print("=" * 78)
    print("合成设定：%d rank × %d 阶段，基准 %.0f ms ± %.0f ms" %
          (n_ranks, n_phases, base_ms, jitter_ms))
    print()

    out = {}
    for rotate, title in [(False, "形态 A：掉队卡固定（rank 3 每阶段都慢 1.6×）"),
                          (True, "形态 B：短板每阶段换一张卡（+%.0f ms 轮转尖峰）" % (jitter_ms * 6))]:
        table = _synth_ranks(n_ranks, n_phases, base_ms, jitter_ms,
                             straggler_rank, straggler_factor, rotate)
        per_phase_max = [max(table[r][p] for r in range(n_ranks)) for p in range(n_phases)]
        per_phase_min = [min(table[r][p] for r in range(n_ranks)) for p in range(n_phases)]
        per_rank_total = [sum(row) for row in table]

        print("-" * 78)
        print(title)
        print("-" * 78)
        print("阶段  " + "".join("   rank%d" % r for r in range(n_ranks)))
        for p in range(n_phases):
            print("  %d   " % p + "".join("%8.1f" % table[r][p] for r in range(n_ranks)))
        print()
        print("  按 rank 求和: " + "  ".join("r%d=%.0f" % (r, per_rank_total[r])
                                              for r in range(n_ranks)))
        sum_max, sum_min = sum(per_phase_max), sum(per_phase_min)
        real_max, real_min = max(per_rank_total), min(per_rank_total)
        print()
        print("  各阶段 max 之和 = %8.1f ms   最慢卡真实值 = %8.1f ms   偏差 %+7.1f ms (%+.1f%%)"
              % (sum_max, real_max, sum_max - real_max, 100.0 * (sum_max / real_max - 1)))
        print("  各阶段 min 之和 = %8.1f ms   最快卡真实值 = %8.1f ms   偏差 %+7.1f ms (%+.1f%%)"
              % (sum_min, real_min, sum_min - real_min, 100.0 * (sum_min / real_min - 1)))
        print("  每个 max 落在哪张卡: " +
              str([max(range(n_ranks), key=lambda r: table[r][p]) for p in range(n_phases)]))
        print("  每个 min 落在哪张卡: " +
              str([min(range(n_ranks), key=lambda r: table[r][p]) for p in range(n_phases)]))
        print()
        out[rotate] = {"sum_max": sum_max, "real_max": real_max,
                       "sum_min": sum_min, "real_min": real_min}

    print("=> 形态 A（掉队卡固定）：各阶段 max 恰好都落在同一张卡上，相加碰巧等于最慢卡真实值；")
    print("   形态 B（短板轮转）：max 来自不同的卡，相加得到一个没有任何卡跑出过的数。")
    print("   所以「把各项 max 加起来」既可能碰巧对，也可能系统性错——取决于短板是否固定。")
    print()
    return out



experiment_minmax_additivity()

实验二  多 rank 的 min/max 不是同一张卡，各项 max 相加没有物理含义
合成设定：8 rank × 6 阶段，基准 100 ms ± 20 ms

------------------------------------------------------------------------------
形态 A：掉队卡固定（rank 3 每阶段都慢 1.6×）
------------------------------------------------------------------------------
阶段     rank0   rank1   rank2   rank3   rank4   rank5   rank6   rank7
  0      100.0   118.4   116.8   184.2   113.5   111.9   110.3   108.7
  1      114.6   113.0   111.3   175.6   108.1   106.5   104.9   103.2
  2      109.2   107.5   105.9   166.9   102.7   101.1   119.4   117.8
  3      103.7   102.1   100.5   190.2   117.3   115.6   114.0   112.4
  4      118.3   116.7   115.1   181.5   111.8   110.2   108.6   107.0
  5      112.9   111.3   109.7   172.9   106.4   104.8   103.2   101.6

  按 rank 求和: r0=659  r1=669  r2=659  r3=1071  r4=660  r5=650  r6=660  r7=651

  各阶段 max 之和 =   1071.3 ms   最慢卡真实值 =   1071.3 ms   偏差    +0.0 ms (+0.0%)
  各阶段 min 之和 =    613.3 ms   最快卡真实值 =    650.1 ms   偏差   -36.8 ms (-5.7%)
  每个 ma

{False: {'sum_max': 1071.2640000000001,
  'real_max': 1071.2640000000001,
  'sum_min': 613.34,
  'real_min': 650.0999999999999},
 True: {'sum_max': 1387.92,
  'real_max': 1071.2640000000001,
  'sum_min': 614.96,
  'real_min': 660.3799999999999}}

> **形态 A（掉队卡固定）**：各阶段 max 恰好都落在同一张卡上，相加碰巧等于最慢卡真实值。
> **形态 B（短板轮转）**：max 来自不同的卡，相加得到一个**没有任何卡跑出过的数**。
>
> 所以「把各项 max 加起来」既可能碰巧对，也可能系统性错——取决于短板是否固定。这不是可以靠约定消除的，只能靠改观测方案：要么采逐 rank 原始值，要么采 top-1 最慢 rank 的整步时长。

## 第 3 步 · `filter(x > 0.0)` 的静默丢弃

**这步在做什么**：没进过该 timer 的 rank 会被 `filter` 掉，于是 min 变成「**活跃子集里的最快**」，与「全体的最快」不是同一个量。

过滤本身是合理的（没进过 timer 不该报 0），**但它不告诉你少了谁**。

In [4]:
# ---- 摘自 tools/profiling_bench/timer_semantics.py ----
def experiment_zero_filter(n_ranks=8, pct=0.25):
    """实验三：`filter(x > 0.0)` 会静默丢掉没进过该 timer 的 rank。"""
    full = [100.0 + 6.0 * r for r in range(n_ranks)]
    skip_ranks = list(range(int(n_ranks * pct)))
    partial = [0.0 if r in skip_ranks else full[r] for r in range(n_ranks)]

    print("=" * 78)
    print("实验三  filter(x > 0.0) 的静默丢弃：min 可能来自一个只干了一点活的 rank")
    print("=" * 78)
    print("场景：基础阶段 %d 个 rank 都进了该 timer；增量阶段只有部分 rank 进入。" % n_ranks)
    print()
    print("  无过滤时按 min 选中的 rank = %d （值 %.1f）"
          % (min(range(n_ranks), key=lambda r: partial[r]),
             min(partial)))
    kept = [v for v in partial if v > 0.0]
    kept_ranks = [r for r in range(n_ranks) if partial[r] > 0.0]
    print("  过滤 0 后按 min 选中的 rank = %d （值 %.1f）"
          % (min(kept_ranks, key=lambda r: partial[r]), min(kept)))
    print()
    print("  被过滤掉的 rank: %s" % skip_ranks)
    print("  过滤后仍在统计里的 rank: %s" % kept_ranks)
    print()
    print("=> 过滤本身是合理的（没进过 timer 不该报 0），但**它不告诉你少了谁**。")
    print("   于是 min 变成「活跃子集里的最快」，与「全体的最快」不是同一个量。")
    print()



experiment_zero_filter()

实验三  filter(x > 0.0) 的静默丢弃：min 可能来自一个只干了一点活的 rank
场景：基础阶段 8 个 rank 都进了该 timer；增量阶段只有部分 rank 进入。

  无过滤时按 min 选中的 rank = 0 （值 0.0）
  过滤 0 后按 min 选中的 rank = 2 （值 112.0）

  被过滤掉的 rank: [0, 1]
  过滤后仍在统计里的 rank: [2, 3, 4, 5, 6, 7]

=> 过滤本身是合理的（没进过 timer 不该报 0），但**它不告诉你少了谁**。
   于是 min 变成「活跃子集里的最快」，与「全体的最快」不是同一个量。



## 第 4 步 · `reset()` 不清 `_active_time`

**这步在做什么**：`timers.py:171` 的 `reset()` 只清 `_elapsed`，而 `_active_time`（`timers.py:212`）从 timer 构造起只增不减。于是两者的比值是一个**占空比**，但只在稳态才成立。

同一个数在早期迭代里完全没有意义——**要拿占空比，必须用稳态窗口内的差值，不能拿累计量直接除。**

In [5]:
# ---- 摘自 tools/profiling_bench/timer_semantics.py ----
def experiment_active_time(n_iter=10, warmup_iter=3, iter_ms=50.0):
    """实验四：`reset()` 不清 `_active_time` -> elapsed/active_time 是占空比。"""
    print("=" * 78)
    print("实验四  reset() 不清 _active_time：elapsed 与 active_time 的比是占空比")
    print("=" * 78)
    elapsed_each = iter_ms
    cum_elapsed = 0.0      # 每次 reset 后归零
    cum_active = 0.0       # 从创建起累计，reset 不影响
    print("  迭代   elapsed(本次)   active_time(累计)   elapsed/active_time")
    for it in range(1, n_iter + 1):
        cum_elapsed = elapsed_each
        cum_active += elapsed_each
        ratio = cum_elapsed / cum_active
        note = "   <- warmup 阶段" if it <= warmup_iter else ""
        print("   %3d      %8.1f ms       %10.1f ms          %6.3f%s"
              % (it, cum_elapsed, cum_active, ratio, note))
    print()
    print("=> 同一个 timer 对象上 elapsed 每次归零、active_time 只增不减，")
    print("   所以早期迭代的 elapsed/active_time 会系统性偏大。")
    print("   想拿占空比必须用**稳态窗口**的差值，不能拿累计量直接除。")
    print()



experiment_active_time()

实验四  reset() 不清 _active_time：elapsed 与 active_time 的比是占空比
  迭代   elapsed(本次)   active_time(累计)   elapsed/active_time
     1          50.0 ms             50.0 ms           1.000   <- warmup 阶段
     2          50.0 ms            100.0 ms           0.500   <- warmup 阶段
     3          50.0 ms            150.0 ms           0.333   <- warmup 阶段
     4          50.0 ms            200.0 ms           0.250
     5          50.0 ms            250.0 ms           0.200
     6          50.0 ms            300.0 ms           0.167
     7          50.0 ms            350.0 ms           0.143
     8          50.0 ms            400.0 ms           0.125
     9          50.0 ms            450.0 ms           0.111
    10          50.0 ms            500.0 ms           0.100

=> 同一个 timer 对象上 elapsed 每次归零、active_time 只增不减，
   所以早期迭代的 elapsed/active_time 会系统性偏大。
   想拿占空比必须用**稳态窗口**的差值，不能拿累计量直接除。



## 小结

内置 timer 报的是「**排空后的时间**」，不是「阶段占用的时间」。三条推论：

1. **overlap 越好 → 边界处在飞的工作越多 → 读数膨胀越大。** 拿它评价重叠优化会得到反向结论。
2. **多 rank 的 min/max 相加没有物理含义**，取决于短板是否固定。
3. **`filter(x > 0.0)` 会静默缩小统计范围**，min 变成「活跃子集里的最快」。

替代方案：CUDA Event 打点（异步、不改变执行），或从 trace 里读区间跨度。

## 源码依据

基准 commit `60e039626`（Megatron-LM v0.20.0，2026-08-21 main）：

- `megatron/core/timers.py:109` `Timer`
- `megatron/core/timers.py:143` / `:156` `start()` / `stop()` 里的 `torch.cuda.synchronize()`
- `megatron/core/timers.py:171` `reset()`（注释就写着 `Don't reset _active_time`）
- `megatron/core/timers.py:212` `active_time`
- `megatron/core/timers.py:318` `_get_global_min_max_time()` 里的 `filter(lambda x: x > 0.0, ...)`